# Standalone DistilBERT Full Fine-Tuning

This notebook runs the standalone minimal DistilBERT full fine-tuning experiment end to end.

It uses only files inside `standalone_distilbert_full_ft/`. It does not import the parent repo's `src/` package.

## What This Notebook Does

1. Checks the GPU runtime.
2. Mounts Google Drive.
3. Clones or updates the repo branch that contains the standalone folder.
4. Installs the standalone dependencies.
5. Logs into W&B from Colab Secrets when available.
6. Shows the hardcoded experiment constants in the Python file.
7. Runs full fine-tuning.
8. Displays saved metrics and sample predictions.
9. Copies outputs to Google Drive for later inspection.

In [ ]:
# Runtime check. Use a GPU runtime for the full experiment.
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
!nvidia-smi

In [ ]:
# Mount Google Drive so outputs can be copied out of the Colab VM.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Clone or update the repo. If this folder later becomes its own repo,
# change REPO_URL and PROJECT_DIR here.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Naaeeen/hate-speech-ft.git"
BRANCH = "standalone-distilbert-full-ft"
PROJECT_DIR = Path("/content/hate-speech-ft")
STANDALONE_DIR = PROJECT_DIR / "standalone_distilbert_full_ft"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(STANDALONE_DIR)
print("Working directory:", Path.cwd())
print("Files:", sorted(path.name for path in STANDALONE_DIR.iterdir()))

In [ ]:
# Install standalone dependencies.
!pip install -q -r requirements.txt

In [ ]:
# W&B login through Colab Secrets.
# Add WANDB_API_KEY for online W&B logging.
# Optional: add HF_TOKEN if Hugging Face rate limits slow dataset/model downloads.
import os

try:
    from google.colab import userdata
    wandb_api_key = userdata.get("WANDB_API_KEY")
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    wandb_api_key = None
    hf_token = None

if wandb_api_key:
    os.environ["WANDB_API_KEY"] = wandb_api_key
    import wandb
    wandb.login(key=wandb_api_key, relogin=True)
    print("W&B login completed.")
else:
    print("No WANDB_API_KEY found in Colab Secrets.")
    print("Either add the secret, run wandb.login(), or set WANDB_MODE='offline' / USE_WANDB=False in the script.")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    try:
        from huggingface_hub import login as hf_login
        hf_login(token=hf_token, add_to_git_credential=False)
        print("Hugging Face login completed.")
    except Exception as exc:
        print(f"HF_TOKEN found, but Hugging Face login failed: {exc}")
else:
    print("No HF_TOKEN found. Public dataset/model downloads should still work, but may be rate-limited.")

## Review Or Edit Experiment Constants

The standalone script intentionally keeps settings as constants in the source file. To change the experiment, edit `train_distilbert_hatexplain.py` directly before running the training cell.

For a smoke test, change `MAX_TRAIN_SAMPLES`, `MAX_EVAL_SAMPLES`, and `NUM_EPOCHS`. For the full run, keep sample caps as `None`.

In [ ]:
# Print the hardcoded settings that usually matter.
from pathlib import Path

script_path = Path("train_distilbert_hatexplain.py")
prefixes = (
    "TRIAL_ID =",
    "SEARCH_STAGE =",
    "SEED =",
    "DATASET_NAME =",
    "MODEL_NAME =",
    "OUTPUT_DIR =",
    "OVERWRITE_OUTPUT_DIR =",
    "MAX_LENGTH =",
    "LEARNING_RATE =",
    "TRAIN_BATCH_SIZE =",
    "EVAL_BATCH_SIZE =",
    "NUM_EPOCHS =",
    "WEIGHT_DECAY =",
    "WARMUP_RATIO =",
    "MAX_GRAD_NORM =",
    "OPTIM =",
    "LR_SCHEDULER_TYPE =",
    "LOGGING_STEPS =",
    "EARLY_STOPPING_PATIENCE =",
    "EARLY_STOPPING_THRESHOLD =",
    "MIXED_PRECISION =",
    "GRADIENT_CHECKPOINTING =",
    "MAX_TRAIN_SAMPLES =",
    "MAX_EVAL_SAMPLES =",
    "MAX_TEST_SAMPLES =",
    "RUN_TEST =",
    "USE_WANDB =",
    "WANDB_PROJECT =",
    "WANDB_ENTITY =",
    "WANDB_MODE =",
    "WANDB_RUN_NAME =",
)

for line in script_path.read_text(encoding="utf-8").splitlines():
    if line.startswith(prefixes):
        print(line)

In [ ]:
# Optional convenience cell for a quick smoke run.
# Leave RUN_THIS_CELL = False for the full experiment.
RUN_THIS_CELL = False

if RUN_THIS_CELL:
    text = script_path.read_text(encoding="utf-8")
    replacements = {
        "TRIAL_ID = \"standalone_distilbert_full_ft_tuning_seed42\"": "TRIAL_ID = \"standalone_distilbert_full_ft_smoke_seed42\"",
        "SEARCH_STAGE = \"tuning\"": "SEARCH_STAGE = \"smoke\"",
        "OUTPUT_DIR = Path(__file__).resolve().parent / \"outputs\" / \"distilbert_full_ft\"": "OUTPUT_DIR = Path(__file__).resolve().parent / \"outputs\" / \"distilbert_full_ft_smoke_seed42\"",
        "MAX_TRAIN_SAMPLES = None": "MAX_TRAIN_SAMPLES = 64",
        "MAX_EVAL_SAMPLES = None": "MAX_EVAL_SAMPLES = 64",
        "MAX_TEST_SAMPLES = None": "MAX_TEST_SAMPLES = 64",
        "NUM_EPOCHS = 3": "NUM_EPOCHS = 1",
        "WANDB_MODE = \"online\"": "WANDB_MODE = \"offline\"",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    script_path.write_text(text, encoding="utf-8")
    print("Smoke-run constants written. Re-run the previous cell to inspect them.")

In [ ]:
# Run the experiment. This will train, evaluate, save metrics/predictions,
# and save the final model under standalone_distilbert_full_ft/outputs/.
!python train_distilbert_hatexplain.py

In [ ]:
# Inspect saved outputs.
import importlib.util
import json
from pathlib import Path

spec = importlib.util.spec_from_file_location("standalone_train", "train_distilbert_hatexplain.py")
standalone_train = importlib.util.module_from_spec(spec)
spec.loader.exec_module(standalone_train)
OUTPUT_DIR = Path(standalone_train.OUTPUT_DIR)
print("Resolved OUTPUT_DIR:", OUTPUT_DIR)
print("Output files:")
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print("-", path)
else:
    print("Output directory does not exist yet.")

metrics_path = OUTPUT_DIR / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("\nMetrics:")
    print(json.dumps(metrics, indent=2, sort_keys=True)[:6000])
else:
    print("metrics.json not found. Check failure.json if the run failed.")

In [ ]:
# Inspect a few validation predictions.
pred_path = OUTPUT_DIR / "predictions_validation.json"
if pred_path.exists():
    preds = json.loads(pred_path.read_text(encoding="utf-8"))
    print("Validation prediction count:", len(preds))
    for row in preds[:3]:
        print(json.dumps(row, indent=2, ensure_ascii=False)[:2000])
else:
    print("predictions_validation.json not found.")

In [ ]:
# Copy outputs to Google Drive. This preserves metrics and predictions after
# the Colab VM shuts down.
import importlib.util
import shutil
from datetime import datetime
from pathlib import Path

if "OUTPUT_DIR" not in globals():
    spec = importlib.util.spec_from_file_location("standalone_train", "train_distilbert_hatexplain.py")
    standalone_train = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(standalone_train)
    OUTPUT_DIR = Path(standalone_train.OUTPUT_DIR)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_root = Path("/content/drive/MyDrive/hate_speech_ft/standalone_distilbert_full_ft")
target_dir = drive_root / f"run_{timestamp}"
target_dir.parent.mkdir(parents=True, exist_ok=True)

if OUTPUT_DIR.exists():
    shutil.copytree(OUTPUT_DIR, target_dir, dirs_exist_ok=True)
    print("Copied outputs to:", target_dir)
else:
    print("No output directory found to copy.")

## Next Manual Experiment

To run a different experiment, edit constants in `train_distilbert_hatexplain.py`, then rerun from the training cell onward.

This notebook is intentionally not a config system. The source file is the experiment record, and each run writes its own JSON snapshots under `outputs/distilbert_full_ft/`.